# CASteer — Alpha Hyperparameter Search (SANA)

Sweep over `alpha` values for SANA steering vectors.
Metrics: **CLIP Score** (text-image alignment) and **FID** (distribution quality) on MS-COCO validation prompts.

**Protocol:**
- Sample ~250 text prompts from MS-COCO val2017 captions
- For each prompt, generate 4 images → ~1 000 images per alpha
- Compute FID against COCO val2017 real images (5K)

**Requirements:** T4 GPU or better. Estimated runtime: ~3–4 hours for 8 alpha values.

In [ ]:
!pip install -q diffusers==0.35.2 transformers==4.57.3 accelerate sentencepiece
!pip install -q open_clip_torch clean-fid
!pip install -q tqdm

In [ ]:
import os
import json
import random
import pickle
import gc
from collections import defaultdict
from pathlib import Path

import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Configuration

In [ ]:
# ============================================================
# Hyperparameter search space
# ============================================================
ALPHAS = [1.2, 2, 5, 8, 10, 15]

# Dynamic alpha decay configurations (tested alongside fixed alphas)
# decay: "linear" | "cosine" | "exp"
DYNAMIC_ALPHA_CONFIGS = [
    {"name": "dynamic_linear_10", "alpha_start": 10, "decay": "linear"},
    {"name": "dynamic_cosine_10", "alpha_start": 10, "decay": "cosine"},
    {"name": "dynamic_exp_10",    "alpha_start": 10, "decay": "exp"},
    {"name": "dynamic_linear_15", "alpha_start": 15, "decay": "linear"},
]

# ============================================================
# Generation settings
# ============================================================
NUM_PROMPTS = 250         # COCO captions to sample
IMAGES_PER_PROMPT = 4     # 250 * 4 = 1000 images per alpha
NUM_STEPS = 20            # SANA denoising steps
HOOK_POINT = "cross_attn" # Hook point for steering
STRATEGY = "all_layers"   # all_layers / late_layers / early_layers / timestep_scaled

# ============================================================
# Steering vectors
# ============================================================
# Set to a .pickle path to skip re-computation (e.g. from Google Drive).
# Leave None to compute from scratch.
SV_BANK_PATH = None

# Concepts used when computing from scratch (multi-concept mode)
CONCEPTS = [
    "tench", "goldfish", "shark", "hen", "ostrich", "brambling",
    "goldfinch", "house finch", "junco", "indigo bunting", "robin",
    "bulbul", "jay", "magpie", "water ouzel", "kite", "bald eagle",
    "vulture", "agama", "chameleon", "iguana", "alligator lizard",
    "triceratops", "scorpion", "tarantula", "centipede", "peacock",
    "flamingo", "pelican", "king penguin", "hummingbird", "toucan",
    "drake", "goose", "black swan", "dalmatian", "poodle", "collie",
    "golden retriever", "labrador", "persian cat", "siamese cat",
    "tabby cat", "tiger", "lion", "panda", "polar bear", "gorilla",
    "chimpanzee", "zebra"
]

# ============================================================
# Paths
# ============================================================
OUTPUT_DIR = "alpha_sweep"
COCO_DIR = "coco"
MODEL_ID = "Efficient-Large-Model/Sana_600M_512px_BF16_diffusers"

# ============================================================
# Reproducibility
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 1. Download MS-COCO val2017

In [ ]:
os.makedirs(COCO_DIR, exist_ok=True)

# Annotations (captions)
if not os.path.exists(f"{COCO_DIR}/annotations/captions_val2017.json"):
    !wget -q --show-progress http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O {COCO_DIR}/annotations.zip
    !unzip -qo {COCO_DIR}/annotations.zip -d {COCO_DIR}/
    !rm {COCO_DIR}/annotations.zip
    print("Annotations downloaded.")
else:
    print("Annotations already present.")

# Validation images (FID reference)
if not os.path.exists(f"{COCO_DIR}/val2017"):
    !wget -q --show-progress http://images.cocodataset.org/zips/val2017.zip -O {COCO_DIR}/val2017.zip
    !unzip -qo {COCO_DIR}/val2017.zip -d {COCO_DIR}/
    !rm {COCO_DIR}/val2017.zip
    print("Val images downloaded.")
else:
    print("Val images already present.")

num_ref_images = len(list(Path(f"{COCO_DIR}/val2017").glob("*.jpg")))
print(f"COCO val2017: {num_ref_images} reference images")

## 2. Sample COCO prompts

In [ ]:
with open(f"{COCO_DIR}/annotations/captions_val2017.json") as f:
    coco_data = json.load(f)

# Take one caption per image (first encountered)
image_to_caption = {}
for ann in coco_data["annotations"]:
    img_id = ann["image_id"]
    if img_id not in image_to_caption:
        image_to_caption[img_id] = ann["caption"]

all_captions = list(image_to_caption.values())
print(f"Unique COCO val images with captions: {len(all_captions)}")

rng_prompts = random.Random(SEED)
selected_prompts = rng_prompts.sample(all_captions, NUM_PROMPTS)

# Save for reproducibility
os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(os.path.join(OUTPUT_DIR, "prompts.json"), "w") as f:
    json.dump(selected_prompts, f, indent=2)

print(f"Selected {NUM_PROMPTS} prompts. Examples:")
for p in selected_prompts[:5]:
    print(f"  • {p}")

## 3. Load SANA pipeline

In [ ]:
from diffusers import SanaPipeline

pipe = SanaPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
pipe.to(device)

num_layers = len(pipe.transformer.transformer_blocks)
print(f"SANA loaded: {num_layers} transformer blocks")

## 4. Controller (steering hook)

In [ ]:
import abc
import math


class VectorControl(abc.ABC):
    def __init__(self):
        self.cur_step = 0
        self.num_att_layers = -1
        self.cur_att_layer = 0

    def reset(self):
        self.cur_step = 0
        self.cur_att_layer = 0

    def between_steps(self):
        return

    @abc.abstractmethod
    def forward(self, vector, layer_idx: int):
        raise NotImplementedError

    def __call__(self, vector, layer_idx: int):
        vector = self.forward(vector, layer_idx)
        self.cur_att_layer += 1
        if self.cur_att_layer == self.num_att_layers:
            self.cur_att_layer = 0
            self.between_steps()
            self.cur_step += 1
        return vector


class SanaVectorStore(VectorControl):
    def __init__(self, steering_vectors=None, steer=True,
                 alpha=10, beta=2, steer_back=False,
                 active_layers=None, timestep_scaling=False,
                 decay_type="linear",
                 total_steps=20, device="cpu"):
        super().__init__()
        self.step_store = {"layers": []}
        self.vector_store = defaultdict(dict)
        self.steering_vectors = steering_vectors
        self.steer = steer
        self.alpha = alpha
        self.beta = beta
        self.steer_back = steer_back
        self.device = device
        self.active_layers = active_layers
        self.timestep_scaling = timestep_scaling
        self.decay_type = decay_type  # "linear" | "cosine" | "exp"
        self.total_steps = total_steps

    def reset(self):
        super().reset()
        self.step_store = {"layers": []}
        self.vector_store = defaultdict(dict)

    def _get_alpha(self):
        if self.timestep_scaling:
            t = self.cur_step / max(self.total_steps - 1, 1)  # 0 → 1
            if self.decay_type == "cosine":
                decay = 0.5 * (1.0 + math.cos(math.pi * t))
            elif self.decay_type == "exp":
                decay = math.exp(-3.0 * t)   # 1 at t=0, ~0.05 at t=1
            else:                            # "linear" (default)
                decay = max(1.0 - t, 0.1)
            return self.alpha * decay
        return self.alpha

    def forward(self, vector, layer_idx: int):
        if self.steer and self.steering_vectors is not None:
            if self.active_layers is None or layer_idx in self.active_layers:
                keys = sorted(self.steering_vectors.keys())
                num_steer = keys[0] if len(keys) == 1 else min(self.cur_step, keys[-1])

                sv = self.steering_vectors[num_steer]["layers"][layer_idx]
                sv = torch.tensor(sv).to(self.device).view(1, 1, -1)
                norm = torch.norm(vector, dim=2, keepdim=True)

                if self.steer_back:
                    sim = torch.tensordot(vector, sv, dims=([2], [2])).view(
                        vector.size()[0], vector.size()[1], 1)
                    sim = torch.where(sim > 0, sim, 0)
                    vector = vector - (self.beta * sim) * sv.expand(1, vector.size()[1], -1)
                else:
                    alpha = self._get_alpha()
                    vector = vector + alpha * sv.expand(1, vector.size()[1], -1)

                vector = vector / torch.norm(vector, dim=2, keepdim=True) * norm

        self.step_store["layers"].append(
            vector.data.cpu().numpy()[len(vector) // 2:].mean(axis=0).mean(axis=0)
        )
        return vector

    def between_steps(self):
        self.vector_store[self.cur_step] = self.step_store
        self.step_store = {"layers": []}


def register_vector_control_sana(model, controller, hook_point="cross_attn"):
    """Patch SANA transformer blocks to route activations through the controller."""
    def block_forward(block, layer_idx):
        def forward(
            hidden_states: torch.Tensor,
            attention_mask=None,
            encoder_hidden_states=None,
            encoder_attention_mask=None,
            timestep=None,
            height: int = None,
            width: int = None,
        ) -> torch.Tensor:
            batch_size = hidden_states.shape[0]
            shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = (
                block.scale_shift_table[None] + timestep.reshape(batch_size, 6, -1)
            ).chunk(6, dim=1)

            norm_hidden_states = block.norm1(hidden_states)
            norm_hidden_states = norm_hidden_states * (1 + scale_msa) + shift_msa
            norm_hidden_states = norm_hidden_states.to(hidden_states.dtype)
            attn_output = block.attn1(norm_hidden_states)
            if hook_point == "self_attn":
                attn_output = controller(attn_output, layer_idx)
            hidden_states = hidden_states + gate_msa * attn_output

            if block.attn2 is not None:
                attn_output = block.attn2(
                    hidden_states,
                    encoder_hidden_states=encoder_hidden_states,
                    attention_mask=encoder_attention_mask,
                )
                if hook_point == "cross_attn":
                    attn_output = controller(attn_output, layer_idx)
                hidden_states = attn_output + hidden_states

            if hook_point == "residual":
                hidden_states = controller(hidden_states, layer_idx)

            norm_hidden_states = block.norm2(hidden_states)
            norm_hidden_states = norm_hidden_states * (1 + scale_mlp) + shift_mlp
            norm_hidden_states = norm_hidden_states.unflatten(1, (height, width)).permute(0, 3, 1, 2)
            ff_output = block.ff(norm_hidden_states)
            ff_output = ff_output.flatten(2, 3).permute(0, 2, 1)
            hidden_states = hidden_states + gate_mlp * ff_output
            return hidden_states

        return forward

    count = 0
    for i, block in enumerate(model.transformer_blocks):
        block.forward = block_forward(block, i)
        count += 1
    controller.num_att_layers = count

## 5. Compute (or load) steering vectors

Multi-concept mode: one prompt per ImageNet class → per-concept bank.
This is a one-time cost (~15–20 min on T4). Set `SV_BANK_PATH` above to skip.

In [ ]:
if SV_BANK_PATH and os.path.exists(SV_BANK_PATH):
    with open(SV_BANK_PATH, "rb") as f:
        concept_bank = pickle.load(f)
    print(f"Loaded concept bank from {SV_BANK_PATH}: {len(concept_bank)} concepts")
else:
    print(f"Computing steering vectors for {len(CONCEPTS)} concepts …")
    prompts_pos = [f"a photo of {cls}" for cls in CONCEPTS]
    prompts_neg = ["a photo"] * len(CONCEPTS)

    pos_vectors, neg_vectors = [], []

    for i, (pp, pn) in enumerate(tqdm(zip(prompts_pos, prompts_neg),
                                       total=len(prompts_pos), desc="Collecting activations")):
        # Positive pass
        ctrl = SanaVectorStore(device=device)
        ctrl.steer = False
        register_vector_control_sana(pipe.transformer, ctrl, hook_point=HOOK_POINT)
        _ = pipe(prompt=pp, num_inference_steps=NUM_STEPS,
                 generator=torch.Generator(device=device).manual_seed(0))
        pos_vectors.append(ctrl.vector_store)

        # Negative pass
        ctrl = SanaVectorStore(device=device)
        ctrl.steer = False
        register_vector_control_sana(pipe.transformer, ctrl, hook_point=HOOK_POINT)
        _ = pipe(prompt=pn, num_inference_steps=NUM_STEPS,
                 generator=torch.Generator(device=device).manual_seed(0))
        neg_vectors.append(ctrl.vector_store)

    # Build per-concept bank
    num_block_layers = len(pos_vectors[0][0]["layers"])
    concept_bank = {}

    for idx, concept in enumerate(CONCEPTS):
        concept_sv = {}
        for step in range(NUM_STEPS):
            concept_sv[step] = {"layers": []}
            for layer in range(num_block_layers):
                pv = pos_vectors[idx][step]["layers"][layer]
                nv = neg_vectors[idx][step]["layers"][layer]
                sv = pv - nv
                norm = np.linalg.norm(sv)
                sv = sv / norm if norm > 0 else sv
                concept_sv[step]["layers"].append(sv)
        concept_bank[concept] = concept_sv

    # Save
    sv_save_path = os.path.join(OUTPUT_DIR, "concept_bank.pickle")
    with open(sv_save_path, "wb") as f:
        pickle.dump(concept_bank, f)
    print(f"Saved concept bank ({len(concept_bank)} concepts) to {sv_save_path}")

    # Free activation memory
    del pos_vectors, neg_vectors
    gc.collect()

concept_names = list(concept_bank.keys())
print(f"Concept bank ready: {len(concept_names)} concepts")

## 6. Pre-assign concepts for reproducibility

Each `(prompt, seed)` pair gets a fixed random concept so the **only** variable across alphas is the steering strength.

In [ ]:
rng_concepts = random.Random(SEED + 1)
concept_assignments = {}
for pi in range(NUM_PROMPTS):
    for seed in range(IMAGES_PER_PROMPT):
        concept_assignments[(pi, seed)] = rng_concepts.choice(concept_names)


def get_active_layers(strategy, n_layers):
    """Return layer indices for the given strategy."""
    if strategy == "all_layers":
        return None
    elif strategy == "late_layers":
        return list(range(n_layers // 2, n_layers))
    elif strategy == "early_layers":
        return list(range(0, n_layers // 2))
    elif strategy == "random_layers":
        n = n_layers // 2
        return sorted(random.sample(range(n_layers), n))
    elif strategy == "timestep_scaled":
        return None
    return None


print(f"Concept assignments ready: {NUM_PROMPTS} prompts × {IMAGES_PER_PROMPT} seeds")

## 7. Alpha sweep — image generation

Images are saved to `alpha_sweep/alpha_<value>/`. Already-generated images are skipped (resume-safe).

In [ ]:
active_layers = get_active_layers(STRATEGY, num_layers)
use_timestep_scaling = (STRATEGY == "timestep_scaled")

# ── Fixed-alpha sweep ─────────────────────────────────────────────────────────
for alpha in ALPHAS:
    alpha_dir = os.path.join(OUTPUT_DIR, f"alpha_{alpha}")
    os.makedirs(alpha_dir, exist_ok=True)

    existing = len(list(Path(alpha_dir).glob("*.png")))
    target = NUM_PROMPTS * IMAGES_PER_PROMPT
    if existing >= target:
        print(f"alpha={alpha}: {existing}/{target} images exist — skipping")
        continue

    print(f"\n{'=' * 50}")
    print(f"Generating for alpha={alpha}  ({existing}/{target} already exist)")
    print(f"{'=' * 50}")

    pbar = tqdm(total=target - existing, desc=f"alpha={alpha}")

    for pi, prompt in enumerate(selected_prompts):
        for seed in range(IMAGES_PER_PROMPT):
            img_path = os.path.join(alpha_dir, f"p{pi:04d}_s{seed}.png")
            if os.path.exists(img_path):
                continue

            if alpha == 0:
                ctrl = SanaVectorStore(device=device)
                ctrl.steer = False
            else:
                concept = concept_assignments[(pi, seed)]
                ctrl = SanaVectorStore(
                    steering_vectors=concept_bank[concept],
                    steer=True,
                    alpha=alpha,
                    active_layers=active_layers,
                    timestep_scaling=use_timestep_scaling,
                    total_steps=NUM_STEPS,
                    device=device,
                )

            register_vector_control_sana(pipe.transformer, ctrl, hook_point=HOOK_POINT)
            img = pipe(
                prompt=prompt,
                num_inference_steps=NUM_STEPS,
                generator=torch.Generator(device=device).manual_seed(seed),
            ).images[0]
            img.save(img_path)
            pbar.update(1)

    pbar.close()
    print(f"alpha={alpha}: done")

# ── Dynamic alpha decay sweep ─────────────────────────────────────────────────
print("\n" + "=" * 50)
print("Dynamic alpha decay configurations")
print("=" * 50)

for cfg in DYNAMIC_ALPHA_CONFIGS:
    cfg_dir = os.path.join(OUTPUT_DIR, cfg["name"])
    os.makedirs(cfg_dir, exist_ok=True)

    existing = len(list(Path(cfg_dir).glob("*.png")))
    target = NUM_PROMPTS * IMAGES_PER_PROMPT
    if existing >= target:
        print(f"{cfg['name']}: {existing}/{target} images exist — skipping")
        continue

    print(f"\n{'-' * 50}")
    print(f"Generating for {cfg['name']}  (α₀={cfg['alpha_start']}, decay={cfg['decay']})")
    print(f"  ({existing}/{target} already exist)")

    pbar = tqdm(total=target - existing, desc=cfg["name"])

    for pi, prompt in enumerate(selected_prompts):
        for seed in range(IMAGES_PER_PROMPT):
            img_path = os.path.join(cfg_dir, f"p{pi:04d}_s{seed}.png")
            if os.path.exists(img_path):
                continue

            concept = concept_assignments[(pi, seed)]
            ctrl = SanaVectorStore(
                steering_vectors=concept_bank[concept],
                steer=True,
                alpha=cfg["alpha_start"],
                active_layers=active_layers,
                timestep_scaling=True,       # always on for dynamic configs
                decay_type=cfg["decay"],
                total_steps=NUM_STEPS,
                device=device,
            )
            register_vector_control_sana(pipe.transformer, ctrl, hook_point=HOOK_POINT)
            img = pipe(
                prompt=prompt,
                num_inference_steps=NUM_STEPS,
                generator=torch.Generator(device=device).manual_seed(seed),
            ).images[0]
            img.save(img_path)
            pbar.update(1)

    pbar.close()
    print(f"{cfg['name']}: done")

print("\nGeneration complete for all configurations!")

## 8. Evaluate — CLIP Score

In [ ]:
import open_clip

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
clip_model = clip_model.to(device).eval()
clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")


def compute_clip_scores_for_dir(image_dir, prompts, images_per_prompt):
    """Return (mean, std) of CLIP cosine similarity over all (image, prompt) pairs."""
    all_scores = []
    for pi, prompt in enumerate(tqdm(prompts, desc="CLIP", leave=False)):
        tokens = clip_tokenizer([prompt]).to(device)
        with torch.no_grad():
            txt_f = clip_model.encode_text(tokens)
            txt_f = F.normalize(txt_f, dim=-1)

        for seed in range(images_per_prompt):
            img_path = os.path.join(image_dir, f"p{pi:04d}_s{seed}.png")
            if not os.path.exists(img_path):
                continue
            img = Image.open(img_path).convert("RGB")
            img_t = clip_preprocess(img).unsqueeze(0).to(device)
            with torch.no_grad():
                img_f = clip_model.encode_image(img_t)
                img_f = F.normalize(img_f, dim=-1)
            score = (img_f @ txt_f.T).item()
            all_scores.append(score)

    return float(np.mean(all_scores)), float(np.std(all_scores))


print("CLIP model loaded.")

## 9. Evaluate — FID

FID is computed between each `alpha_<value>/` folder and COCO val2017 real images using **clean-fid**.

In [ ]:
from cleanfid import fid

# Pre-compute Inception stats for COCO val2017 (cached after first run)
coco_ref_dir = os.path.join(COCO_DIR, "val2017")
print(f"FID reference dir: {coco_ref_dir}")
print(f"Reference images: {len(list(Path(coco_ref_dir).glob('*.jpg')))}")

## 10. Run full evaluation

In [ ]:
results = {}

# ── Fixed-alpha configs ───────────────────────────────────────────────────────
for alpha in ALPHAS:
    alpha_dir = os.path.join(OUTPUT_DIR, f"alpha_{alpha}")
    n_images = len(list(Path(alpha_dir).glob("*.png")))
    print(f"\nEvaluating alpha={alpha}  ({n_images} images)")

    clip_mean, clip_std = compute_clip_scores_for_dir(
        alpha_dir, selected_prompts, IMAGES_PER_PROMPT
    )
    print(f"  CLIP Score: {clip_mean:.4f} ± {clip_std:.4f}")

    fid_score = fid.compute_fid(alpha_dir, coco_ref_dir)
    print(f"  FID:        {fid_score:.2f}")

    results[f"alpha_{alpha}"] = {
        "label": f"α={alpha}",
        "clip_score_mean": clip_mean,
        "clip_score_std": clip_std,
        "fid": fid_score,
        "num_images": n_images,
        "kind": "fixed",
    }

# ── Dynamic alpha configs ─────────────────────────────────────────────────────
for cfg in DYNAMIC_ALPHA_CONFIGS:
    cfg_dir = os.path.join(OUTPUT_DIR, cfg["name"])
    n_images = len(list(Path(cfg_dir).glob("*.png")))
    print(f"\nEvaluating {cfg['name']}  ({n_images} images)")

    clip_mean, clip_std = compute_clip_scores_for_dir(
        cfg_dir, selected_prompts, IMAGES_PER_PROMPT
    )
    print(f"  CLIP Score: {clip_mean:.4f} ± {clip_std:.4f}")

    fid_score = fid.compute_fid(cfg_dir, coco_ref_dir)
    print(f"  FID:        {fid_score:.2f}")

    results[cfg["name"]] = {
        "label": f"α₀={cfg['alpha_start']} {cfg['decay']}",
        "clip_score_mean": clip_mean,
        "clip_score_std": clip_std,
        "fid": fid_score,
        "num_images": n_images,
        "kind": "dynamic",
    }

# Save
results_path = os.path.join(OUTPUT_DIR, "results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to {results_path}")

## 11. Results — Table

In [ ]:
fixed_keys   = [k for k, v in results.items() if v["kind"] == "fixed"]
dynamic_keys = [k for k, v in results.items() if v["kind"] == "dynamic"]

print(f"{'Config':>28} | {'CLIP Score':>16} | {'FID':>8}")
print("-" * 60)

print("Fixed alpha:")
for key in fixed_keys:
    r = results[key]
    print(f"  {r['label']:>26} | {r['clip_score_mean']:.4f} ± {r['clip_score_std']:.4f} | {r['fid']:>8.2f}")

print("\nDynamic alpha (decay from α₀):")
for key in dynamic_keys:
    r = results[key]
    print(f"  {r['label']:>26} | {r['clip_score_mean']:.4f} ± {r['clip_score_std']:.4f} | {r['fid']:>8.2f}")

# Best across all configs (CLIP↑ + FID↓ trade-off)
all_keys = list(results.keys())
clips = np.array([results[k]["clip_score_mean"] for k in all_keys])
fids  = np.array([results[k]["fid"]             for k in all_keys])

clip_norm = (clips - clips.min()) / (clips.max() - clips.min() + 1e-8)
fid_norm  = (fids  - fids.min())  / (fids.max()  - fids.min()  + 1e-8)
combined  = clip_norm + (1 - fid_norm)

best_key   = all_keys[int(np.argmax(combined))]
best_label = results[best_key]["label"]
print(f"\n→ Best config (CLIP↑ + FID↓ trade-off): {best_label}")
print(f"  CLIP={results[best_key]['clip_score_mean']:.4f}, FID={results[best_key]['fid']:.2f}")

## 12. Results — Plots

In [ ]:
import math as _math

fixed_keys   = [k for k, v in results.items() if v["kind"] == "fixed"]
dynamic_keys = [k for k, v in results.items() if v["kind"] == "dynamic"]

fixed_clips  = [results[k]["clip_score_mean"] for k in fixed_keys]
fixed_stds   = [results[k]["clip_score_std"]  for k in fixed_keys]
fixed_fids   = [results[k]["fid"]             for k in fixed_keys]
fixed_labels = [results[k]["label"]           for k in fixed_keys]

dyn_clips  = [results[k]["clip_score_mean"] for k in dynamic_keys]
dyn_stds   = [results[k]["clip_score_std"]  for k in dynamic_keys]
dyn_fids   = [results[k]["fid"]             for k in dynamic_keys]
dyn_labels = [results[k]["label"]           for k in dynamic_keys]

decay_marker = {"linear": "s", "cosine": "D", "exp": "^"}
decay_color  = {"linear": "tab:orange", "cosine": "tab:green", "exp": "tab:red"}

best_key = max(results, key=lambda k: (
    (results[k]["clip_score_mean"] - clips.min()) / (clips.max() - clips.min() + 1e-8)
    + 1 - (results[k]["fid"] - fids.min()) / (fids.max() - fids.min() + 1e-8)
))

# ── Scatter: CLIP vs FID (all configs) ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(fixed_fids, fixed_clips, marker="o", s=80, color="tab:blue",
           label="Fixed α", zorder=3)
for x, y, lbl in zip(fixed_fids, fixed_clips, fixed_labels):
    ax.annotate(lbl, (x, y), textcoords="offset points", xytext=(5, 4), fontsize=8)

seen_decay = set()
for k, cx, cy, lbl in zip(dynamic_keys, dyn_fids, dyn_clips, dyn_labels):
    decay_type = next(c["decay"] for c in DYNAMIC_ALPHA_CONFIGS if c["name"] == k)
    legend_lbl = f"Dynamic {decay_type}" if decay_type not in seen_decay else "_nolegend_"
    seen_decay.add(decay_type)
    ax.scatter(cx, cy, marker=decay_marker[decay_type], s=100,
               color=decay_color[decay_type], label=legend_lbl, zorder=3)
    ax.annotate(lbl, (cx, cy), textcoords="offset points", xytext=(5, 4), fontsize=8)

ax.scatter(results[best_key]["fid"], results[best_key]["clip_score_mean"],
           s=220, marker="*", color="gold", edgecolors="black", linewidths=0.8,
           zorder=5, label=f"Best: {results[best_key]['label']}")

ax.set_xlabel("FID  (lower = better)", fontsize=12)
ax.set_ylabel("CLIP Score  (higher = better)", fontsize=12)
ax.set_title("Fixed vs Dynamic Alpha — CLIP Score vs FID", fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "alpha_sweep_scatter.png"), dpi=150, bbox_inches="tight")
plt.show()

# ── Bar charts: CLIP and FID side by side ────────────────────────────────────
all_keys_plot   = fixed_keys + dynamic_keys
all_labels_plot = fixed_labels + dyn_labels
all_clips_plot  = fixed_clips  + dyn_clips
all_stds_plot   = fixed_stds   + dyn_stds
all_fids_plot   = fixed_fids   + dyn_fids
bar_colors = ["tab:blue"] * len(fixed_keys) + ["tab:orange"] * len(dynamic_keys)

x = np.arange(len(all_keys_plot))
fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

ax1.bar(x, all_clips_plot, yerr=all_stds_plot, capsize=4, color=bar_colors, alpha=0.8)
ax1.set_xticks(x)
ax1.set_xticklabels(all_labels_plot, rotation=30, ha="right", fontsize=9)
ax1.set_ylabel("CLIP Score  (↑)", fontsize=12)
ax1.set_title("CLIP Score per Configuration", fontsize=13)
ax1.axvline(len(fixed_keys) - 0.5, color="gray", linestyle="--", alpha=0.5)
ax1.grid(True, axis="y", alpha=0.3)

ax2.bar(x, all_fids_plot, color=bar_colors, alpha=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels(all_labels_plot, rotation=30, ha="right", fontsize=9)
ax2.set_ylabel("FID  (↓)", fontsize=12)
ax2.set_title("FID per Configuration", fontsize=13)
ax2.axvline(len(fixed_keys) - 0.5, color="gray", linestyle="--", alpha=0.5)
ax2.grid(True, axis="y", alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor="tab:blue", label="Fixed α"),
                   Patch(facecolor="tab:orange", label="Dynamic α")]
ax1.legend(handles=legend_elements, fontsize=9)

plt.tight_layout()
fig2.savefig(os.path.join(OUTPUT_DIR, "alpha_sweep_bars.png"), dpi=150, bbox_inches="tight")
plt.show()

# ── Decay schedule visualisation ─────────────────────────────────────────────
ts = np.linspace(0, 1, NUM_STEPS)
fig3, ax3 = plt.subplots(figsize=(7, 4))
ax3.plot(ts, np.ones_like(ts),                        label="fixed",     color="tab:blue",   linestyle="--")
ax3.plot(ts, np.maximum(1.0 - ts, 0.1),               label="linear",    color="tab:orange")
ax3.plot(ts, 0.5 * (1 + np.cos(np.pi * ts)),          label="cosine",    color="tab:green")
ax3.plot(ts, np.exp(-3.0 * ts),                       label="exp (k=3)", color="tab:red")
ax3.set_xlabel("Normalised timestep  t = step / (T−1)", fontsize=11)
ax3.set_ylabel("α(t) / α₀", fontsize=11)
ax3.set_title("Alpha Decay Schedules", fontsize=13)
ax3.legend()
ax3.grid(True, alpha=0.3)
plt.tight_layout()
fig3.savefig(os.path.join(OUTPUT_DIR, "decay_schedules.png"), dpi=150, bbox_inches="tight")
plt.show()

## 13. (Optional) Save to Google Drive

In [ ]:
# Uncomment to copy results to Drive
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r {OUTPUT_DIR}/results.json {OUTPUT_DIR}/alpha_sweep*.png /content/drive/MyDrive/